# Customer Churn Prediction – EDA and model comparison
Training for deployment lives in `train.py`; this notebook explores the data and compares models fairly (each on the same test set).

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import train  # reuses the exact preprocessing used in production

## 1. Load and inspect

In [ ]:
df = pd.read_csv(train.DATA_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')  # blanks -> NaN
print(df.shape)
print(df.isna().sum()[lambda s: s > 0])
df.describe()

## 2. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x='Churn', data=df, ax=axes[0])
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[1])
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, ax=axes[2])
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 3.5))
sns.barplot(x='Contract', y=(df['Churn'] == 'Yes').astype(int), data=df)
plt.ylabel('Churn rate'); plt.title('Churn rate by contract'); plt.show()

In [ ]:
sns.heatmap(df.select_dtypes('number').corr(), annot=True, cmap='Spectral'); plt.show()

## 3. Model comparison (same split, same preprocessing)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X, y = train.load_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
models = {
    'Logistic Regression': [StandardScaler(), LogisticRegression(max_iter=2000)],
    'Decision Tree': [DecisionTreeClassifier(max_depth=6, random_state=42)],
    'Random Forest': [RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5, class_weight='balanced', random_state=42)],
}
rows = {}
for name, steps in models.items():
    pipe = Pipeline([('prep', train.build_preprocessor(X)), *[(f's{i}', s) for i, s in enumerate(steps)]])
    pipe.fit(X_train, y_train)
    rows[name] = train.evaluate(pipe, X_test, y_test)
pd.DataFrame(rows).T.round(3)

## 4. Feature importance of the deployed model

In [ ]:
import joblib
model = joblib.load(train.MODEL_PATH)
importance = pd.Series(model['model'].feature_importances_, index=model['prep'].get_feature_names_out()).sort_values()
importance.plot.barh(figsize=(8, 6), title='Feature importance'); plt.show()